# Homework 11: Bootstrap, CLT, and Normal Approximation (Problems 7 - 10)

Lukas Vrouwenvelder

In [189]:
import pandas as pd
import numpy as np


## Problem 7

In [190]:
ames = pd.read_csv('./data/ames_prices.csv') # Load data
ames

,Order,PID,area,price,MS.SubClass,MS.Zoning,Lot.Frontage,Lot.Area,Street,Alley,...,Screen.Porch,Pool.Area,Pool.QC,Fence,Misc.Feature,Misc.Val,Mo.Sold,Yr.Sold,Sale.Type,Sale.Condition
0,1,526301100,1656,215000,20,RL,141.0,31770,Pave,NaN,...,0,0,NaN,NaN,NaN,0,5,2010,WD,Normal
1,2,526350040,896,105000,20,RH,80.0,11622,Pave,NaN,...,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
2,3,526351010,1329,172000,20,RL,81.0,14267,Pave,NaN,...,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
3,4,526353030,2110,244000,20,RL,93.0,11160,Pave,NaN,...,0,0,NaN,NaN,NaN,0,4,2010,WD,Normal
4,5,527105010,1629,189900,60,RL,74.0,13830,Pave,NaN,...,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2925,2926,923275080,1003,142500,80,RL,37.0,7937,Pave,NaN,...,0,0,NaN,GdPrv,NaN,0,3,2006,WD,Normal
2926,2927,923276100,902,131000,20,RL,NaN,8885,Pave,NaN,...,0,0,NaN,MnPrv,NaN,0,6,2006,WD,Normal
2927,2928,923400125,970,132000,85,RL,62.0,10441,Pave,NaN,...,0,0,NaN,MnPrv,Shed,700,7,2006,WD,Normal
2928,2929,924100070,1389,170000,20,RL,77.0,10010,Pave,NaN,...,0,0,NaN,NaN,NaN,0,4,2006,WD,Normal


In [191]:
# create an indicator that equals 1 when Central.Air == 'Y' and 0 otherwise
ames['central_air'] = (ames['Central.Air'] == 'Y').astype(int)

# estimate proportion or homes with central air conditioning by taking the mean of the indicator variable
print("Proportion of homes with central air conditioning:", ames['central_air'].mean())

Proportion of homes with central air conditioning: 0.9331058020477816


In [192]:
# Using a reproducible random-number seed, generate at least 2,000 bootstrap estimates of this proportion. Report the seed and number of bootstrap replications.
seed = 42
n_bootstrap = 2000
rng = np.random.default_rng(seed)

x = ames['central_air'].to_numpy()
n = len(x)

# resample n homes with replacement and take the mean of the indicator each time
boot_props = np.array([rng.choice(x, size=n, replace=True).mean() for i in range(n_bootstrap)])

print("Seed:", seed)
print("Number of bootstrap replications:", n_bootstrap)
print("Mean of bootstrap estimates:", boot_props.mean())

Seed: 42
Number of bootstrap replications: 2000
Mean of bootstrap estimates: 0.933342662116041


In [193]:
# Compute the bootstrap standard error as the sample standard deviation of the bootstrap estimates (using ddof=1 ensures that there's the $1/(N-1)$ degrees of freedom correction)
# ddof=1 applies the 1/(N-1) degrees-of-freedom correction
boot_se = np.std(boot_props, ddof=1)
print("Bootstrap standard error:", boot_se)

Bootstrap standard error: 0.004590333440769273


In [194]:
# Using the original estimate and the bootstrap standard error, construct Normal 90%, 95%, and 99% confidence intervals.
original_estimate = ames['central_air'].mean()

# Normal interval: estimate +/- z * bootstrap SE
z_values = {90: 1.645, 95: 1.96, 99: 2.576}
for level, z in z_values.items():
    lower = original_estimate - z * boot_se
    upper = original_estimate + z * boot_se
    print(f"{level}% Normal CI: ({lower:.4f}, {upper:.4f})")

90% Normal CI: (0.9256, 0.9407)
95% Normal CI: (0.9241, 0.9421)
99% Normal CI: (0.9213, 0.9449)


Use each confidence interval to evaluate $H_0:p=.90$. At the matching significance levels $\alpha=.10,.05,.01$, state whether you reject or fail to reject the null hypothesis. Briefly interpret your results.

1) 90% Normal CI: (0.9256, 0.9407): Does not include .90; Reject the H_0. 
2) 95% Normal CI: (0.9241, 0.9421): Does not include .90; Reject the H_0. 
3) 99% Normal CI: (0.9213, 0.9449): Does not include .90; Reject the H_0. 

Based on these findings we reject the null at all confidence levels tested. The share of homes with central air is likely at about 93%, where all the CIs are centered and where the bootstrap mean is (.933). 

## Problem 8

In [195]:
# estimate the Pearson correlation between np.arcsinh(area) and np.arcsinh(price)
correlation = np.corrcoef(np.arcsinh(ames['area']), np.arcsinh(ames['price']))[0, 1]
correlation

np.float64(0.723166153009136)

In [196]:
# Using a reproducible random-number seed, generate at least 2,000 bootstrap estimates of the correlation by resampling paired rows. Report the seed and number of bootstrap replications.
seed = 42
n_bootstrap = 2000
rng = np.random.default_rng(seed)

area_t = np.arcsinh(ames['area']).to_numpy()
price_t = np.arcsinh(ames['price']).to_numpy()
n = len(area_t)

# resample row indices with replacement so each home's (area, price) pair stays together
boot_corrs = np.empty(n_bootstrap)
for b in range(n_bootstrap):
    idx = rng.integers(0, n, size=n)
    boot_corrs[b] = np.corrcoef(area_t[idx], price_t[idx])[0, 1]

bs_avg = boot_corrs.mean()

print("Seed:", seed)
print("Number of bootstrap replications:", n_bootstrap)
print("Mean of bootstrap estimates:", bs_avg)

Seed: 42
Number of bootstrap replications: 2000
Mean of bootstrap estimates: 0.723189849593135


In [197]:
# Compute the bootstrap standard error as the sample standard deviation of the bootstrap estimates (using ddof=1 ensures that there's the $1/(N-1)$ degrees of freedom correction).
boot_se_corr = np.std(boot_corrs, ddof=1)
print("Bootstrap standard error:", boot_se_corr)

Bootstrap standard error: 0.010285986570021804


In [198]:
# Normal interval: estimate +/- z * bootstrap SE
z_values = {90: 1.645, 95: 1.96, 99: 2.576}
for level, z in z_values.items():
    lower = correlation - z * boot_se_corr
    upper = correlation + z * boot_se_corr
    print(f"{level}% Normal CI: ({lower:.4f}, {upper:.4f})")

90% Normal CI: (0.7062, 0.7401)
95% Normal CI: (0.7030, 0.7433)
99% Normal CI: (0.6967, 0.7497)


In [199]:
# Use each confidence interval to evaluate $H_0:\rho=0$. At the matching significance levels $\alpha=.10,.05,.01$, state whether you reject or fail to reject the null hypothesis. 
# Briefly interpret the association without making a causal claim.

for level, z in z_values.items():
    lower = correlation - z * boot_se_corr
    upper = correlation + z * boot_se_corr
    if lower <= 0 <= upper:
        print(f"At {level}% confidence level, we fail to reject the null hypothesis H0: correlation =0.")
    else:
        print(f"At {level}% confidence level, we reject the null hypothesis H0: correlation =0.")

At 90% confidence level, we reject the null hypothesis H0: correlation =0.
At 95% confidence level, we reject the null hypothesis H0: correlation =0.
At 99% confidence level, we reject the null hypothesis H0: correlation =0.


None of the 90%, 95%, or 99% intervals contain 0, so we reject H_0: correlation =0 at alpha=.10,.05,.01. The estimated correlation between `arcsinh(area)` and `arcsinh(price)` is about 0.72, a strong positive association. In this sample, homes with more living area tend to sell for higher prices. This is an association only. The correlation does not show that adding area causes a higher price, since other factors (neighborhood, quality, age, lot size) could have influence on pricing as well.

The correlation measures how `area` and `price` move together within the same home. Resampling rows keeps each home's (area, price) pair together, so bootstrap sample estimates vary around the observed correlation. Resampling separately, area would've matched prices with areas from unrelated home, making the correlation between price and area random and somewhat meaningless.

## Problem 9

In [200]:
metabric = pd.read_csv('./data/metabric.csv')

# check for missing values in Overall Survival (Months)
metabric['Overall Survival (Months)'].isnull().sum() #none

np.int64(0)

In [201]:
original_estimate = metabric['Overall Survival (Months)'].mean()

print('Average Overall Survival in Months:', original_estimate)

Average Overall Survival in Months: 129.09429138744105


In [202]:
# Using a reproducible random-number seed, generate at least 2,000 bootstrap estimates of the mean. Report the seed and number of bootstrap replications.
seed = 42
n_bootstrap = 2000
rng = np.random.default_rng(seed)

x = metabric['Overall Survival (Months)'].to_numpy()
n = len(x)

# resample n survival times with replacement and take the mean each time
boot_means = np.array([rng.choice(x, size=n, replace=True).mean() for i in range(n_bootstrap)])

bs_avg_mean = boot_means.mean()

print("Seed:", seed)
print("Number of bootstrap replications:", n_bootstrap)
print("Mean of bootstrap estimates:", bs_avg_mean)

Seed: 42
Number of bootstrap replications: 2000
Mean of bootstrap estimates: 129.11665899726978


In [203]:
boot_se = np.std(boot_means, ddof=1)
print("Bootstrap standard error:", boot_se)

Bootstrap standard error: 2.135806001210453


In [204]:
# Normal interval: estimate +/- z * bootstrap SE
z_values = {90: 1.645, 95: 1.96, 99: 2.576}
for level, z in z_values.items():
    lower = original_estimate - z * boot_se
    upper = original_estimate + z * boot_se
    print(f"{level}% Normal CI: ({lower:.4f}, {upper:.4f})")

90% Normal CI: (125.5809, 132.6077)
95% Normal CI: (124.9081, 133.2805)
99% Normal CI: (123.5925, 134.5961)


- alpha= .1; 90% Normal CI: (125.5809, 132.6077); excludes 120; reject H_o. 
- alpha= .05; 95% Normal CI: (124.9081, 133.2805); excludes 120; reject H_o. 
- alpha= .01; 99% Normal CI: (123.5925, 134.5961); excludes 120; reject H_o.

In [205]:
print('Average Overall Survival in Years:', original_estimate/12)
print('H_o in Years:', 120/12)

z_values = {90: 1.645, 95: 1.96, 99: 2.576}
for level, z in z_values.items():
    lower = (original_estimate - z * boot_se)/12
    upper = (original_estimate + z * boot_se)/12
    print(f"{level}% Normal CI in Years: ({lower:.4f}, {upper:.4f})")

Average Overall Survival in Years: 10.757857615620088
H_o in Years: 10.0
90% Normal CI in Years: (10.4651, 11.0506)
95% Normal CI in Years: (10.4090, 11.1067)
99% Normal CI in Years: (10.2994, 11.2163)


*Interpret the estimate and confidence intervals in both months and years. Because living patients' eventual death times are right-censored, explain why these results do not estimate unrestricted mean lifetime.*


The 99% confidence interval is the widest, most inclusive CI computed, and it ranges from 10.3 to 11.2yrs (123.5925, 134.5961 months) for survival time centered around the original estimate (129.1 months ≈ 10.8 years). The null hypothesis that the true mean is 10 years (120 months) is not supported by the evidence; we reject the claim that the true average survival time is 10 years bases on these data at alpha=.01 (99% CI), which means that at any alpha higher than .01, we would also reject the null hypothesis.

The data, however, are right-censored. The survival for a patient could represent the patients actual survival (they died within the observation window) or that the patient survived past the observational period, in which case the survival time represents the elapsed time between the patient's first appointment, and their last follow-up. 

The results mix these two values, and the result is a number that almost certainly underestimates average survival time. 

## Problem 10

In [206]:
nhanes = pd.read_csv("./data/nhanes_data_17_18.csv")
nhanes 

/var/folders/j5/w2f07zxx2y1_2zk3yh9_5_x80000gn/T/ipykernel_44743/1055581574.py:1: DtypeWarning: Columns (0: ServedActiveDutyInUsArmedForces) have mixed types. Specify dtype option on import or set low_memory=False.
  nhanes = pd.read_csv("./data/nhanes_data_17_18.csv")


,SEQN,GeneralHealthCondition,EverBreastfedOrFedBreastmilk,AgeStoppedBreastfeedingdays,AgeFirstFedFormuladays,AgeStoppedReceivingFormuladays,AgeStartedOtherFoodbeverage,AgeFirstFedMilkdays,TypeOfMilkFirstFedWholeMilk,TypeOfMilkFirstFed2Milk,...,DaysSmokedCigsDuringPast30Days,AvgCigarettesdayDuringPast30Days,TriedToQuitSmoking,TimesStoppedSmokingCigarettes,HowLongWereYouAbleToStopSmoking,UnitOfMeasureDayweekmonthyear_2_SMQ,CurrentSelfreportedHeightInches,CurrentSelfreportedWeightPounds,TriedToLoseWeightInPastYear,TimesLost10LbsOrMoreToLoseWeight
0,93703.0,NaN,1.0,273.0,1.0,365.0,152.0,365.0,Whole or regular milk,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,93704.0,NaN,1.0,60.0,3.0,365.0,126.0,365.0,Whole or regular milk,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,93705.0,Good,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,63.0,165.0,0.0,11 times or more
3,93706.0,Very good,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,68.0,145.0,0.0,Never
4,93707.0,Good,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8361,102952.0,Very good,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,117.0,0.0,Never
8362,102953.0,Fair or,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,65.0,218.0,0.0,Never
8363,102954.0,Good,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,66.0,150.0,0.0,Never
8364,102955.0,Very good,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [207]:
# search for 'age' in columns and print the resulting column names
nhanes.columns[nhanes.columns.str.contains('Age')]

# This is the one we want: AgeInYearsAtScreening

Index(['AgeStoppedBreastfeedingdays', 'AgeFirstFedFormuladays',
       'AgeStoppedReceivingFormuladays', 'AgeStartedOtherFoodbeverage',
       'AgeFirstFedMilkdays', 'HowOftenDrankMilkAge512',
       'HowOftenDrankMilkAge1317', 'HowOftenDrankMilkAge1835',
       'AgeWhenToldYouHadHeartAttack', 'AgeWhenToldYouHadAStroke',
       'Age1StCancerMcq230ADiagnosed', 'AgeInYearsAtScreening',
       'AgeInMonthsAtScreening0To24Mos', 'AgeWhenFirstToldYouHadDiabetes',
       'AgeStartedSmokingCigarettesRegularly'],
      dtype='str')

In [208]:
# select respondents who are at least 18 years old and have an observed value for SleepHoursWeekdaysOrWorkdays. 
nhanes_filtered = nhanes[(nhanes['AgeInYearsAtScreening'] >= 18) & (nhanes['SleepHoursWeekdaysOrWorkdays'].notnull())]

In [209]:
# Estimate their mean weekday sleep.
sleep_means = nhanes_filtered['SleepHoursWeekdaysOrWorkdays'].mean()
print("Mean weekday sleep (filtered data):", sleep_means)

Mean weekday sleep (filtered data): 7.62124248496994


In [210]:
# Using a reproducible random-number seed, generate at least 2,000 bootstrap estimates of the mean. Report the seed and number of bootstrap replications.
seed = 42
n_bootstrap = 2000
rng = np.random.default_rng(seed)

x = nhanes_filtered['SleepHoursWeekdaysOrWorkdays'].to_numpy()
n = len(x)
print("Number of respondents in filtered data:", n)

# resample n sleep hours with replacement and take the mean each time
boot_means = np.array([rng.choice(x, size=n, replace=True).mean() for i in range(n_bootstrap)])

bs_avg_mean = boot_means.mean()

print("Seed:", seed)
print("Number of bootstrap replications:", n_bootstrap)
print("Mean of bootstrap estimates:", bs_avg_mean)

Number of respondents in filtered data: 5489
Seed: 42
Number of bootstrap replications: 2000
Mean of bootstrap estimates: 7.620444343231918


In [211]:
# Compute the bootstrap standard error as the sample standard deviation of the bootstrap estimates (ddof=1).
boot_se = np.std(boot_means, ddof=1)
print("Bootstrap standard error:", boot_se)

Bootstrap standard error: 0.022750666141160076


In [212]:
# Using the original estimate and the bootstrap standard error, construct centered-Normal 90%, 95%, and 99% confidence intervals.

# Normal interval: estimate +/- z * bootstrap SE
z_values = {90: 1.645, 95: 1.96, 99: 2.576}
for level, z in z_values.items():
    lower = sleep_means - z * boot_se
    upper = sleep_means + z * boot_se
    print(f"{level}% Normal CI: ({lower:.4f}, {upper:.4f})")

90% Normal CI: (7.5838, 7.6587)
95% Normal CI: (7.5767, 7.6658)
99% Normal CI: (7.5626, 7.6798)


In [213]:
# Use each confidence interval to evaluate $H_0:\mu=8$ hours. At the matching significance levels $\alpha=.10,.05,.01$, state whether you reject or fail to reject the null hypothesis.
for level, z in z_values.items():
    lower = sleep_means - z * boot_se
    upper = sleep_means + z * boot_se
    if lower <= 8 <= upper:
        print(f"At {level}% confidence level, we fail to reject the null hypothesis H0: mu=8 hours.")
    else:
        print(f"At {level}% confidence level, we reject the null hypothesis H0: mu=8 hours.")

At 90% confidence level, we reject the null hypothesis H0: mu=8 hours.
At 95% confidence level, we reject the null hypothesis H0: mu=8 hours.
At 99% confidence level, we reject the null hypothesis H0: mu=8 hours.


Standard error is small (.02) in our bootstrapped calculation, partly because `n` is large (5489) in this sample. This leads to narrow confidence intervals in our hypothesis testing. This does not reduce the statistical significance of our findings (even a small difference can be statistically significant); the null hypothesis is still rejected at all confidence levels tested. However, the significance of difference between the mean (7.62) and the null hypothesis (8) is likely fairly small (roughly 23min) in ordinary circumstances. Practically, this difference may not matter much in the 'real world'

These results treat respondents as independent, unweighted observations and ignore NHANES's complex survey design, so they describe this sample rather than the U.S. adult population, and the intervals are not survey-adjusted. This further undermines the reliability of the confidence intervals calculated. They are likely narrower in this resampling experiment than they would be if accounting for survey design. 